# Demonstração do pipeline RAG — Voice of Customer (Olist)

Este notebook percorre, etapa por etapa, o caminho de uma pergunta até a resposta, usando os
mesmos módulos da aplicação (`src/voc_rag`). Pré-requisito: base e índices já construídos
(`python -m voc_rag.data_prep` e `python -m voc_rag.indexing`).

In [1]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.max_colwidth", 110)

from voc_rag import config
from voc_rag.query import plan_query
from voc_rag.pipeline import VoCPipeline
from voc_rag.generation import build_context

## 1. Base de conhecimento
Resumo do que `data_prep.py` produziu, incluindo as verificações de integridade.

In [2]:
report = json.loads(config.DATA_REPORT_FILE.read_text())
print("Avaliações no CSV:", report["total_reviews"], "| com texto:", report["reviews_with_text"],
      "| documentos finais:", report["documents"])
pd.Series(report["integrity"], name="ocorrências").to_frame()

Avaliações no CSV: 99224 | com texto: 42687 | documentos finais: 42114


,ocorrências
avaliacoes_sem_pedido,0
pedidos_sem_cliente,0
itens_sem_produto,0
pedidos_sem_itens,775
scores_fora_de_1_a_5,0
comentarios_com_quebra_de_linha,3852
review_ids_em_mais_de_um_pedido,789
pedidos_com_mais_de_uma_avaliacao,547
pedidos_com_mais_de_um_item,9803
produtos_sem_categoria,610


In [3]:
docs = pd.read_parquet(config.DOCUMENTS_FILE)
docs[["text", "review_score", "purchase_date", "category", "delivery_status", "delay_days"]].sample(5, random_state=7)

,text,review_score,purchase_date,category,delivery_status,delay_days
25939,Material do produto muito inferior do que é exposto no site. Não é o que eu esperava. Sem acabamento também.,3,2018-02-07,moveis_decoracao,no_prazo,-21.0
12535,"Super recomendo, entrega rápida como sempre.",5,2017-07-19,beleza_saude,no_prazo,-11.0
29885,"Eu recomendo esta loja porque eu conheço,mas estou aguardando mercadoria.",3,2018-05-08,beleza_saude,atrasado,7.0
11534,"Comprei vai fazer dois meses, paguei o frete e não entregaram, vou no Procon.. não é a primeira vez que vc...",1,2017-01-09,moveis_decoracao,no_prazo,0.0
29780,não recebi meus produtos!,1,2018-03-28,automotivo,nao_entregue,NaN


In [4]:
# Relação entre situação da entrega e nota (motiva o uso do metadado de entrega)
docs.groupby("delivery_status")["review_score"].agg(["count", "mean"]).round(2)

,count,mean
delivery_status,,
atrasado,3809,1.94
nao_entregue,1867,1.52
no_prazo,36438,4.00


## 2. Transformação da pergunta
`plan_query` produz o plano de consulta: intenção, filtros inferidos, expansão léxica e classificação ampla × focada.

In [5]:
pergunta = "O que os clientes relatam sobre problemas relacionados à entrega?"
plan = plan_query(pergunta)
plan.as_dict()

{'original': 'O que os clientes relatam sobre problemas relacionados à entrega?',
 'normalized': 'o que os clientes relatam sobre problemas relacionados a entrega?',
 'dense_query': 'O que os clientes relatam sobre problemas relacionados à entrega?',
 'sparse_query': 'O que os clientes relatam sobre problemas relacionados à entrega? entrega prazo atraso atrasou chegou chegar recebi receber correios transportadora frete rapido demorou',
 'filters': 'nota 1–3',
 'intent': 'negativo',
 'in_scope': True,
 'core': ['entreg'],
 'broad': False,
 'scope_reason': '',
 'applied_rules': ['intenção negativa -> filtro nota <= 3',
  'expansão BM25 com sinônimos de: entrega']}

## 3. Recuperação
Busca híbrida (densa + BM25) com filtros, fusão RRF, verificação de suporte e MMR.

In [6]:
pipe = VoCPipeline()   # usa o provedor configurado no .env; sem chave, cai no modo extrativo
resp = pipe.ask(pergunta)
print("modo:", resp.retrieval.mode, "| população filtrada:", resp.retrieval.population, "|", resp.retrieval.reason)
pd.DataFrame([{"ref": e.ref, "nota": e.score, "entrega": e.delivery_status, "sim_densa": e.dense_sim,
               "rank_bm25": e.bm25_rank, "texto": e.text} for e in resp.evidences])

modo: focado | população filtrada: 14438 | 94 avaliações sustentam a pergunta; 10 selecionadas como evidência.


,ref,nota,entrega,sim_densa,rank_bm25,texto
0,R1,1,atrasado,0.688,9.0,Atraso na entrega
1,R2,1,atrasado,0.640,12.0,Atrasou na entrega
2,R3,1,atrasado,0.643,39.0,Produto com entrega atrasada
3,R4,1,atrasado,0.196,3.0,Entrega atrasada sera que é correios em greve?
4,R5,3,no_prazo,0.289,4.0,Chegou atrasado mais se desculparam e fizeram a entrega.
5,R6,3,no_prazo,0.612,20.0,Atrasou a entrega
6,R7,1,no_prazo,0.696,NaN,"Em partes. Sou cliente há anos, porém sempre há divergências e problemas na entrega, consta como produto e..."
7,R8,3,atrasado,0.431,1.0,Demora na entrega e ainda atrasou.
8,R9,1,atrasado,0.717,NaN,"sem comentarios, muito confusa a informação sobre entrega"
9,R10,1,no_prazo,0.704,NaN,muitos desencontros nas entregas de produtos


## 4. Contexto enviado ao modelo de linguagem

In [7]:
print(resp.context)

FILTROS APLICADOS: nota 1–3
AVALIAÇÕES NA POPULAÇÃO FILTRADA: 14438

EVIDÊNCIAS:
[R1] (nota 1/5 | compra 2018-02-06 | categoria utilidades_domesticas | entregue com 14 dia(s) de atraso | UF SP | +7 avaliação(ões) com texto idêntico)
"Atraso na entrega"
[R2] (nota 1/5 | compra 2018-04-25 | categoria esporte_lazer | entregue com 3 dia(s) de atraso | UF BA)
"Atrasou na entrega"
[R3] (nota 1/5 | compra 2017-11-25 | categoria cama_mesa_banho | entregue com 11 dia(s) de atraso | UF DF)
"Produto com entrega atrasada"
[R4] (nota 1/5 | compra 2018-02-27 | categoria brinquedos | entregue com 6 dia(s) de atraso | UF SP)
"Entrega atrasada sera que é correios em greve?"
[R5] (nota 3/5 | compra 2017-10-20 | categoria telefonia | entregue no prazo | UF MA)
"Chegou atrasado mais se desculparam e fizeram a entrega."
[R6] (nota 3/5 | compra 2017-07-23 | categoria eletronicos | entregue no prazo | UF RS)
"Atrasou a entrega"
[R7] (nota 1/5 | compra 2018-06-28 | categoria cool_stuff | entregue no prazo | U

## 5. Resposta gerada e validação das citações

In [8]:
print("gerador:", resp.generator, "\n")
print(resp.answer)
print("\nvalidação:", {k: v for k, v in resp.citation_check.items() if k != "answer"})

gerador: llm:gemini/gemini-3.5-flash 

Os clientes relatam predominantemente atrasos e demoras na entrega dos produtos, além de problemas com informações confusas sobre o status do pedido e recebimento parcial de itens.

**Principais pontos identificados:**

* **Atraso e demora na entrega:** Há múltiplos relatos explícitos de entregas com atraso ou demora [R1], [R2], [R3], [R4], [R5], [R6], [R8]. Segundo as estatísticas de temas baseadas em palavras-chave, "Produto não recebido (até a data da avaliação)" representa 40,0% (3227) e "Atraso / demora na entrega" representa 22,5% (1812) das menções.
* **Informações confusas e status divergente:** Os consumidores mencionam desencontros nas entregas [R10], informações confusas sobre o processo [R9] e divergências onde o status consta como "entregue", mas o cliente não recebeu o produto completo [R7].
* **Pedidos incompletos:** Existe relato de recebimento de apenas 1 item do pedido quando o status marcava como entregue [R7]. Nas estatísticas 

## 6. Pergunta ampla: decomposição por tema
Sem termo de conteúdo na pergunta, as evidências são buscadas por tema: duas para cada um dos quatro temas mais frequentes na população filtrada.

In [9]:
resp2 = pipe.ask("Quais são os principais problemas relatados pelos clientes em suas avaliações?")
print("modo:", resp2.retrieval.mode, "| filtros:", resp2.plan.filters.describe())
display(pd.DataFrame(resp2.retrieval.theme_stats_population))
pd.DataFrame([{"ref": e.ref, "tema": e.subquery, "nota": e.score, "texto": e.text} for e in resp2.evidences])

modo: amplo | filtros: nota 1–3


,theme,label,count,pct
0,nao_recebido,Produto não recebido (até a data da avaliação),3845,26.6
1,atraso_na_entrega,Atraso / demora na entrega,1896,13.1
2,troca_reembolso,"Troca, devolução, cancelamento ou reembolso",1742,12.1
3,produto_errado,Produto errado / diferente do anunciado,1173,8.1
4,defeito_avaria,Defeito / produto quebrado ou avariado,917,6.4
5,itens_faltando,Pedido incompleto / itens faltando,827,5.7
6,qualidade_ruim,Qualidade abaixo do esperado,481,3.3
7,custo_beneficio,Preço / custo-benefício,298,2.1


,ref,tema,nota,texto
0,R1,Produto não recebido (até a data da avaliação),1,"ainda recomendo. Se o produto não chegou, como pode ser obrigatório o preenchimento da avaliação geral ou ..."
1,R2,Produto não recebido (até a data da avaliação),1,Produto não entregue = consumidor insatisfeito. Preciso de retorno do stark se o produto ainda será entreg...
2,R3,Atraso / demora na entrega,1,produto insatisfatorio. demora na entrega e produto de baixa qualidade
3,R4,Atraso / demora na entrega,2,"Falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi t..."
4,R5,"Troca, devolução, cancelamento ou reembolso",1,"compra efetuada,debito autorizado ,pedido cancelado sem explicações.Extrema falta de respeito e consideraç..."
5,R6,"Troca, devolução, cancelamento ou reembolso",1,Produto errado. Entrega do produto diferente do solicitado Aguardo orientação para a troca
6,R7,Produto errado / diferente do anunciado,1,"Produto errado,cor e tamanho errado, péssima qualidade."
7,R8,Produto errado / diferente do anunciado,1,Produto incorreto. Produto diferente do anunciado


## 7. Perguntas sem evidência suficiente
A recusa ocorre antes da chamada ao modelo de linguagem.

In [10]:
for q in ["Qual a capital da França?", "Qual foi o faturamento da Olist em 2018?"]:
    r = pipe.ask(q)
    print(q, "->", r.generator, "\n  ", r.answer, "\n")

Qual a capital da França? -> guardrail 
   Não há evidências suficientes na base de avaliações para responder. Motivo: Apenas 0 avaliação(ões) da base têm relação verificável com a pergunta (mínimo exigido: 3). Nenhum termo do domínio de e-commerce/avaliações foi identificado na pergunta. Reformule a pergunta com um tema presente em avaliações de clientes (entrega, qualidade, atendimento, produto, embalagem, troca etc.) ou ajuste os filtros. 

Qual foi o faturamento da Olist em 2018? -> guardrail 
   Não há evidências suficientes na base de avaliações para responder. Motivo: A pergunta exige dados transacionais/financeiros agregados, que não estão nos textos das avaliações (base de conhecimento desta solução). Reformule a pergunta com um tema presente em avaliações de clientes (entrega, qualidade, atendimento, produto, embalagem, troca etc.) ou ajuste os filtros. 



## 8. Filtros de metadados
Filtros explícitos (ou inferidos) são aplicados dentro da busca vetorial e do BM25.

In [11]:
resp3 = pipe.ask("O que os clientes mais elogiam em produtos de beleza?")
print(resp3.plan.filters.describe(), "| população:", resp3.retrieval.population)
pd.DataFrame([{"ref": e.ref, "nota": e.score, "categoria": e.category, "texto": e.text} for e in resp3.evidences]).head(6)

nota 4–5; categoria beleza_saude | população: 2416


,ref,nota,categoria,texto
0,R1,5,beleza_saude,"Qualifico como ótima compra. Recomendo a todos. Entrega feita no prazo, produto maravilhoso."
1,R2,5,beleza_saude,"Entrega antes do prazo, produto de alta qualidade."
2,R3,4,beleza_saude,recomendo tanto a loja como o produto.gostei
3,R4,5,beleza_saude,"Produto muito bom,a loja tbem recomendo a todos que queiram adquirir o produto"
4,R5,4,beleza_saude,muito bom o produto super recomendo o vendedor
5,R6,5,beleza_saude,"Tudo veio do jeito descrito, produto de ótima qualidade, recomendo vendedor."
